<span style="color:red;font-size:2em;font-weight:bold"> PARTIE 2 - Analyse exploratoire</span>

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Modules </span>

In [1]:
# Roots
import gc
import numpy as np
import pandas as pd
# import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Stats
from scipy.stats import zscore, chi2_contingency, f_oneway, chi2

In [2]:
# Sert à éviter les Warnings avec les transformations sur des vues en transformant 
# ces warning en erreur obligeant ainsi à ne travailler que sur des copies ou les originaux.

pd.set_option('mode.chained_assignment','raise')

In [3]:
# Module pour recharger un module sans redemarrer le kernel
# import importlib
%load_ext autoreload
%autoreload 2

In [4]:
# Ajoute le dossier datas_manipulation au sys.path. Remarque ne pas oublier le __init__.py dans le dossier datas_manipulation
import sys
# root_path = Path(__file__).resolve().parents[1] # Ne fonctionne pas sur notebook
root_path = Path.cwd().parent
sys.path.append(str(root_path))


In [ ]:
# Fonctions personnelles
from notebooks.datas_manipulation.convert_datas import convert_csv_to_parquet
from notebooks.datas_manipulation.import_datas import import_datas
from notebooks.datas_manipulation.quick_clean_datas import (
    remove_duplicates,drop_empty_columns, drop_col_with_unique_value,
    clean_infinites, drop_empty_rows
)
from notebooks.datas_manipulation.datas_assembler import assemble_data
from notebooks.datas_manipulation.memory_optimizer import optimize_dtypes, log_metrics

from notebooks.utils.feature_aggregator import agg_features

In [6]:
# Paramètres globaux

# Création dossier results
save_path = root_path.joinpath('datas/results')
Path.mkdir(save_path,exist_ok = True)

# choix de sauvegarde pour la data Xy
yes_choice = {'YES','yes','y','Y'}
save_datas = "n"

# Choix de sauvegarde pour les graphes
save_graphs = False

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Datasets </span>

In [7]:
# Chemin du dossier de données brut
datas_path = root_path /'datas'/'raw_datas'/'Projet+Mise+en+prod+-+home-credit-default-risk'

L'encodage des fichiers n'étant pas uniquement en utf-8 (ASCII) et le poids des données étant conséquente, il a fallut dans un premier temps déterminer le dtype puis les optimiser pour cela, on a choisi de changer le format en parquet. Les avantages sont:

- Compression intelligente : En stockant par colonne (stockage binaire VS stockage en texte brut), Parquet peut compresser les données de façon optimale. Par exemple, si une colonne ne contient que des "0" ou "1", il ne les écrit pas des millions de fois, il utilise des algorithmes comme le Run-Length Encoding.

- Types préservés : Le CSV oublie si une colonne est un float32 ou un int16. Le Parquet, lui, enregistre le schéma. Vous n'aurez plus jamais besoin de chardet ou de redétecter les types au chargement.

- Lecture sélective : Si vous voulez juste la colonne TARGET et SK_ID_CURR, Pandas ne lira que ces octets sur le disque sans charger le reste du fichier. Le CSV, lui, oblige à tout lire ligne par ligne.

En contrepartie, les fichiers parquets ne peuvent pas etre ouvert par de simple éditeur de texte (binaire) et la création des fichiers parquet va solliciter les CPU le temps de compresser et créer les fichiers.

Autres bénéfices par la suite, c'est le format standard en ML.

<span style="color:blue;font-weight:bold"> Conversion au format parquet </span>

In [8]:
# Fonctionnelle mais beaucoup trop gourmande en mémoire
# dfs = [import_csv(file_path) for file_path in datas_path.glob("*.csv")]

# Conversion des csv en parquet + optimisation des dtypes pour éco RAM sauf SK_ID* ==> acceptable
[convert_csv_to_parquet(file_path) for file_path in datas_path.glob("*.csv")
    if ('HomeCredit' not in file_path.name) and ('sample' not in file_path.name)
]

[False, False, False, False, False, False, False, False]

In [9]:
# Vérifie le chemin de chargement
# print(datas_path)
# print(datas_path.exists())
# if datas_path.exists():
#     print(list(datas_path.iterdir()))

<span style="color:blue;font-weight:bold"> Tour d'horizon du contenu des fichiers et déscription des features présentes </span>

In [10]:
# fichier de description
df_description = pd.read_csv(datas_path/"HomeCredit_columns_description.csv",encoding='latin1')
df_description

,Unnamed: 0,Table,Row,Description,Special
0,1,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample,NaN
1,2,application_{train|test}.csv,TARGET,Target variable (1 - client with payment diffi...,NaN
2,5,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving,NaN
3,6,application_{train|test}.csv,CODE_GENDER,Gender of the client,NaN
4,7,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car,NaN
...,...,...,...,...,...
214,217,installments_payments.csv,NUM_INSTALMENT_NUMBER,On which installment we observe payment,NaN
215,218,installments_payments.csv,DAYS_INSTALMENT,When the installment of previous credit was su...,time only relative to the application
216,219,installments_payments.csv,DAYS_ENTRY_PAYMENT,When was the installments of previous credit p...,time only relative to the application
217,220,installments_payments.csv,AMT_INSTALMENT,What was the prescribed installment amount of ...,NaN


In [11]:
print(f"nombre de features uniques:{df_description["Row"].nunique()}")
duplicated_features = df_description.loc[df_description["Row"].duplicated()].reset_index(drop=True)
duplicated_features_list = duplicated_features["Row"].unique().tolist()
print(duplicated_features_list)
display(duplicated_features)

nombre de features uniques:196
['SK_ID_CURR', 'AMT_ANNUITY', 'SK_BUREAU_ID', 'MONTHS_BALANCE', 'SK_ID_PREV ', 'NAME_CONTRACT_STATUS', 'SK_DPD', 'SK_DPD_DEF', 'NAME_CONTRACT_TYPE', 'AMT_CREDIT', 'AMT_GOODS_PRICE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'NAME_TYPE_SUITE']


,Unnamed: 0,Table,Row,Description,Special
0,125,bureau.csv,SK_ID_CURR,ID of loan in our sample - one loan in our sam...,hashed
1,141,bureau.csv,AMT_ANNUITY,Annuity of the Credit Bureau credit,NaN
2,142,bureau_balance.csv,SK_BUREAU_ID,Recoded ID of Credit Bureau credit (unique cod...,hashed
3,146,POS_CASH_balance.csv,SK_ID_CURR,ID of loan in our sample,NaN
4,147,POS_CASH_balance.csv,MONTHS_BALANCE,Month of balance relative to application date ...,time only relative to the application
5,153,credit_card_balance.csv,SK_ID_PREV,ID of previous credit in Home credit related t...,hashed
6,154,credit_card_balance.csv,SK_ID_CURR,ID of loan in our sample,hashed
7,155,credit_card_balance.csv,MONTHS_BALANCE,Month of balance relative to application date ...,time only relative to the application
8,173,credit_card_balance.csv,NAME_CONTRACT_STATUS,"Contract status (active signed,...) on the pre...",NaN
9,174,credit_card_balance.csv,SK_DPD,DPD (Days past due) during the month on the pr...,NaN


On commence par le fichier descriptive des features:

on a 220 lignes correspondant aux features présentes dans tous les fichiers de données et 5 colonnes dans l'ordre suivant: n° de la feature, fichier concerné, le nom de la feature, la définition de la feature puis un espace commentaire (ex: "arrondi","normalisé","hash"...).

En se basant sur la colonne Row (nom de la feature, on trouve qu'il y a 196 features uniques) et pour ce qui des features répétés:

['SK_ID_CURR', 'AMT_ANNUITY', 'SK_BUREAU_ID', 'MONTHS_BALANCE', 'SK_ID_PREV ', 'NAME_CONTRACT_STATUS', 'SK_DPD', 'SK_DPD_DEF', 'NAME_CONTRACT_TYPE', 'AMT_CREDIT', 'AMT_GOODS_PRICE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'NAME_TYPE_SUITE']

On y retrouve notamment les colonnes relationnelles entre les blocs: ['SK_ID_CURR', 'SK_BUREAU_ID', 'SK_ID_PREV '] si on se réfère à la source. 
<span style="color:red">Remarque: erreur dans le nommage SK_BUREAU_ID dans le fichier de description ==> dans la data c'est bien SK_ID_BUREA</span>

On retrouve une description des fichiers ainsi qu'un diagramme de relation sur https://www.kaggle.com/c/home-credit-default-risk/data.


<span style="color:blue;font-weight:bold"> Visualisation du contenu des fichiers et optimisation </span>

<span style="color:blue"> Application train/test </span>

On va dans un premier temps merger ensemble train et test verticalement afin d'avoir le set complet en remplissant en NaN la partie TARGET de test et on créée une colonne supplémentaire "subset" pour se souvenir d'où vient chacune des lignes. **On évitera ainsi un traitement différent entre le train et le test et on pourra ensuite re-splitter correctement les deux jeux si nécéssaire.**

In [12]:
# Seuil de Nan au dela duquel on supprime une colonne
miss = 0.7
miss_percent = int(miss * 100)

In [ ]:
# Chrgement et assemblage de train/test
train_test_path = datas_path / "train_test.parquet"
if not train_test_path.exists():
    df_train_test = assemble_data(datas_path,['application_train.parquet','application_test.parquet'])
    # Réduction de la précision (Gain RAM)
    df_train_test = optimize_dtypes(df_train_test)
    # Sauvegarde du fichier nettoyé
    df_train_test.to_parquet(train_test_path, index=False)

<span style="color:blue"> Nettoyage et optimisation du reste </span>

In [ ]:
# On va créer une liste des fichiers qui n'ont pas encore été nettoyer
files_to_clean = []
for file in datas_path.glob("*.parquet"):
    # On ignore les fichiers déjà nettoyés ainsi que les fichiers train OU test
    if ((f"{miss_percent}_cleaned" in file.name)
        or (file.stem.endswith("_train"))
        or (file.stem.endswith("_test"))
    ):
        continue
    # On restreint la liste aux fichiers ne possédant pas une version nettoyée
    cleaned_version = file.with_name(f"{file.stem}{miss_percent}_cleaned.parquet")
    if cleaned_version.exists():
        continue
    
    files_to_clean.append(file)

In [ ]:
# initialisation de la lite des résultas
analysis_results = []

# for file in datas_path.glob("*.parquet"):
#     if ((f"{miss_percent}_cleaned" in file.name)
#         or ('application_train.parquet' in file.name)
#         or ('application_test.parquet' in file.name)
#     ):
#         continue
for file in files_to_clean:    
    new_file_path = datas_path / file.name.replace(".parquet",f"{miss_percent}_cleaned.parquet")
    
    # Chargement temporaire
    temp_df = pd.read_parquet(file)
    
    # Calcul des métriques AVANT NETTOYAGE
    stage0, mem_mb0, null_count0 = log_metrics(temp_df, "uncleaned")
    
    # Nettoyage des doublons parfaits
    temp_df, dropped_rows = remove_duplicates(temp_df)
    # Suppression des colonnes avec beaucoup de NaN
    temp_df, dropped_cols = drop_empty_columns(temp_df, threshold=miss)
    # Suppression des colonnes a variance nulle
    temp_df, null_var = drop_col_with_unique_value(temp_df)
    
    # Calcul des métriques APRES NETTOYAGE
    stage1, mem_mb1, null_count1 = log_metrics(temp_df, "cleaned")
    
    # Stockage dans un dictionnaire
    file_info = {
        "Fichier": file.name,
        "Lignes": temp_df.shape[0],
        "Colonnes": temp_df.shape[1],
        "RAM (MB) avant nettoyage": round(mem_mb0, 2),
        "RAM (MB) sauvée": round(mem_mb0 - mem_mb1, 2),
        "Doublons supprimés (lignes)": dropped_rows,
        "Cellule NA (total) supprimée": int(null_count0 - null_count1),
        # "% Cellule NA restant": round((null_count0 / total_cells) * 100, 2) if total_cells > 0 else 0,
        f"Col supprimées": dropped_cols + null_var
    }
    
    analysis_results.append(file_info)
    
    # Sauvegarde du fichier nettoyé
    if not new_file_path.exists():
        temp_df.to_parquet(new_file_path, index=False)

    # df_chunk
    print(f"Echantillon du fichier {file.name}")
    display((temp_df.head(10)).T)
    
    # Nettoyage
    del temp_df
    gc.collect()

Echantillon du fichier installments_payments.parquet


,0,1,2,3,4,5,6,7,8,9
SK_ID_PREV,1054186.00,1330831.000,2085231.0,2452527.00,2714724.000,1137312.000,2234264.000,1818599.000,2723183.00,1413990.00
SK_ID_CURR,161674.00,151639.000,193053.0,199697.00,167756.000,164489.000,184693.000,111420.000,112102.00,109741.00
NUM_INSTALMENT_VERSION,1.00,0.000,2.0,1.00,1.000,1.000,4.000,2.000,0.00,1.00
NUM_INSTALMENT_NUMBER,6.00,34.000,1.0,3.00,2.000,12.000,11.000,4.000,14.00,4.00
DAYS_INSTALMENT,-1180.00,-2156.000,-63.0,-2418.00,-1383.000,-1384.000,-349.000,-968.000,-197.00,-570.00
DAYS_ENTRY_PAYMENT,-1187.00,-2156.000,-63.0,-2426.00,-1366.000,-1417.000,-352.000,-994.000,-197.00,-609.00
AMT_INSTALMENT,6948.36,1716.525,25425.0,24350.13,2165.040,5970.375,29432.295,17862.165,70.74,14308.47
AMT_PAYMENT,6948.36,1716.525,25425.0,24350.13,2160.585,5970.375,29432.295,17862.165,70.74,14308.47


Echantillon du fichier POS_CASH_balance.parquet


,0,1,2,3,4,5,6,7,8,9
SK_ID_PREV,1803195,1715348,1784872,1903291,2341044,2207092,1110516,1387235,1220500,2371489
SK_ID_CURR,182943.0,367990.0,397406.0,269225.0,334279.0,342166.0,204376.0,153211.0,112740.0,274851.0
MONTHS_BALANCE,-31,-33,-32,-35,-35,-32,-38,-35,-31,-32
CNT_INSTALMENT,48.0,36.0,12.0,48.0,36.0,12.0,48.0,36.0,12.0,24.0
CNT_INSTALMENT_FUTURE,45.0,35.0,9.0,42.0,35.0,12.0,43.0,36.0,12.0,16.0
NAME_CONTRACT_STATUS,Active,Active,Active,Active,Active,Active,Active,Active,Active,Active
SK_DPD,0,0,0,0,0,0,0,0,0,0
SK_DPD_DEF,0,0,0,0,0,0,0,0,0,0


Echantillon du fichier bureau.parquet


...

Echantillon du fichier credit_card_balance.parquet


,0,1,2,3,4,5,6,7,8,9
SK_ID_PREV,2562384,2582071,1740877,1389973,1891521,2646502,1079071,2095912,2181852,1235299
SK_ID_CURR,378907,363914,371185,337855,126868,380010,171320,118650,367360,203885
MONTHS_BALANCE,-6,-1,-7,-4,-1,-7,-6,-7,-4,-5
AMT_BALANCE,56.97,63975.555,31815.225,236572.11,453919.455,82903.815,353451.645,47962.125,291543.075,201261.195
AMT_CREDIT_LIMIT_ACTUAL,135000,45000,450000,225000,450000,270000,585000,45000,292500,225000
AMT_DRAWINGS_ATM_CURRENT,0.0,2250.0,0.0,2250.0,0.0,0.0,67500.0,45000.0,90000.0,76500.0
AMT_DRAWINGS_CURRENT,877.5,2250.0,0.0,2250.0,11547.0,0.0,67500.0,45000.0,289339.425,111026.7
AMT_DRAWINGS_OTHER_CURRENT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AMT_DRAWINGS_POS_CURRENT,877.5,0.0,0.0,0.0,11547.0,0.0,0.0,0.0,199339.425,34526.7
AMT_INST_MIN_REGULARITY,1700.325,2250.0,2250.0,11795.76,22924.89,4449.105,14684.175,0.0,130.5,6338.34


Echantillon du fichier previous_application.parquet


,0,1,2,3,4,5,6,7,8,9
SK_ID_PREV,2030495,2802425,2523466,2819243,1784265,1383531,2315218,1656711,2367563,2579447
SK_ID_CURR,271877.0,108129.0,122040.0,176158.0,202054.0,199383.0,175704.0,296299.0,342292.0,334349.0
NAME_CONTRACT_TYPE,Consumer loans,Cash loans,Cash loans,Cash loans,Cash loans,Cash loans,Cash loans,Cash loans,Cash loans,Cash loans
AMT_ANNUITY,1730.43,25188.615,15060.735,47041.335,31924.395,23703.93,NaN,NaN,NaN,NaN
AMT_APPLICATION,17145.0,607500.0,112500.0,450000.0,337500.0,315000.0,0.0,0.0,0.0,0.0
AMT_CREDIT,17145.0,679671.0,136444.5,470790.0,404055.0,340573.5,0.0,0.0,0.0,0.0
AMT_DOWN_PAYMENT,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AMT_GOODS_PRICE,17145.0,607500.0,112500.0,450000.0,337500.0,315000.0,NaN,NaN,NaN,NaN
WEEKDAY_APPR_PROCESS_START,SATURDAY,THURSDAY,TUESDAY,MONDAY,THURSDAY,SATURDAY,TUESDAY,MONDAY,MONDAY,SATURDAY
HOUR_APPR_PROCESS_START,15,11,11,7,9,8,11,7,15,15


,Fichier,Lignes,Colonnes,RAM (MB) avant nettoyage,RAM (MB) sauvée,Doublons supprimés (lignes),Cellule NA (total) supprimée,Col supprimées
0,installments_payments.parquet,13605401,8,596.86,0.00,0,-12,[]
5,credit_card_balance.parquet,3840312,23,483.44,-25.64,0,0,[]
1,POS_CASH_balance.parquet,10001358,8,286.14,-66.77,0,-16,[]
3,bureau_balance.parquet,27299925,3,260.35,-182.25,0,0,[]
6,previous_application.parquet,1670214,32,191.15,-133.79,0,-959364,"[RATE_INTEREST_PRIMARY, RATE_INTEREST_PRIVILEG..."
2,bureau.parquet,1716428,16,137.50,-21.28,0,1226783,[AMT_ANNUITY]
4,train_test.parquet,356255,123,125.37,-32.27,0,-129303,[]


In [ ]:
# Récap du nettoyage si réalisé
if analysis_results:
    # Création du DataFrame global
    global_info_df = pd.DataFrame(analysis_results)

    # Tri par occupation RAM pour identifier les fichiers critiques
    global_info_df = global_info_df.sort_values(by="RAM (MB) avant nettoyage", ascending=False)

    # Affichage propre
    display(global_info_df)

On va aborder deux thématique ici: les différents fichiers disponibles avec leur contenu et le nettoyage/optimisation de ces fichiers que l'on aura réaliser.

**Contenu des fichiers**

En comptant le fichier descriptif, on a 10 fichiers (c'est plus que le diagramme car train|test ensemble tandis que le fichier *HomeCredit_columns_description* et *sample_submission* non plus car l'un décrit les features et l'autre). Les fichiers sont reliés par le SK_ID_CURR (l'ID client CHEZ HOME CREDIT) et une liaison complémentaire se fait avec les autres ID (SK_ID_BUREAU pour bureau et bureau_balance / SK_ID_PREV pour previous_application avec POS_CASH_balance, installments_payments et credit_card_balance). Pour ce qui est de leur contenu:
- **application_train/test**: Les données principales (Démographie, revenus, montant du prêt). Une ligne = Un prêt.
- **bureau**: Données de tous les emprunts enregistrés au Bureau du Crédit (toutes les institutions incluant aussi Home Credit).
- **bureau_balance**: Historique mensuel des crédits (état de remboursement).
- **previous_application**: Toutes les demandes de prêts faites par le client chez Home Credit par le passé.
- **POS_CASH_balance**: Historique mensuel des soldes des anciens prêts (Point of Sale et Cash).
- **installments_payments**: Historique de chaque paiement réalisé par rapport aux anciens prêts (réussi ou echoué).
- **credit_card_balance**: Historique mensuel de l'utilisation des cartes de crédit du client.

**Nettoyage/optimiation**

Comme mentionné précédemment, les fichiers ne sont pas exploitables et joignable en l'état (on a près de 3Go de données soit plus de 8Go de RAM), il est donc nécéssaire de restructurer la donnée dans un premier temps.
- On a converti les csv en parquet et on a downcasté les dtypes quand c'est possible sans pertes d'information
- On a converti les XNA/XAP (not available et not applicable) en NaN (Dans le fil de discussion, les organisateurs ont dit que cela signifiait que la donnée était indisponible) de même 365234 pour les durées temporelles ($\approx 1000 ans$)
- On a tenté de supprimer les lignes doublons parfaits (aucun...)
- On a supprimé les colonnes avec un taux de manque de données de plus de 70% (c'est assez aggressif mais je suis parti du fait que les projets en général de OC se concentrent sur le raisonnement et la logique des choix plutot que sur l'aspect technique et ici, vu que la donnée fait crashé le pc...De plus le but est de mettre en place un modèle de scoring et de l'optimiser pas d'avoir le meilleur modèle grâce au meilleur jeu de données.)
- <span style="color:red">On avait voulu a la base convertir les object en category afin de pouvoir rapidement optimiser la mémoire mais étant un format trop rigide pour la partie exploratoire, nous avons décider de l'enlever le temps de finir l'analyse et nous le ferons avant de passer à l'entrainement</span>

La donnée reste tros dense (**c'est une situation commune ou singulière à moi?**) il faut donc traiter les fichiers séparemment.

en considérant le nombre de ligne dans test et train on a environ 355000 lignes (avec une séparation $\approx 85/15$) or on peut voir que les autres fichiers en contiennent BEAUCOUP plus ($> 2.7*10^7 \approx 76$ lignes par client) ce qui nous fait dire que soit pour une même personne, plusieurs lignes leur sont attribuée dans ces fichiers, soit ce sont des doublons (aucun doublons parfait par contre) (peut-être les deux).

On va commencer par l'embranchement bureau/bureau_balance qui concerne l'historique des emprunts renseignés au sein de Credit Bureau.

<span style="color:black;font-size:1em;background-color:yellow"> CREDIT BUREAU EST UNE INSTITUTION QUI RECENSE LES PRETS CONTRAIREMENT A HOME CREDIT QUI EST LA BANQUE PRETEUSE ICI!!!</span>

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Traitement de la branche bureau</span>

Comme il a été dit précédemment au vu du nombre de lignes qui dépasse très largement le nombre d'observations train/test, en partant du postulat que ce dépassement (de plusieurs millions tout de même pour certains fichiers) est majoritairement lié au fait que plusieurs lignes concernent un même client (avec éventuellement un peu de doublons), on va devoir regarder séparemment leur contenu et aggreger afin de n'avoir qu'une observation par client.

- **bureau** contient l'ensemble des prêts déclarés dans une institution des clients. Chaque ligne correspond a un crédit et caractérisée par deux ID:
    - SK_ID_CURR: l'ID du client chez Home Credit
    - SK_ID_BUREAU: L'ID d'un prêt enregistré chez Credit Bureau
- **bureau_balance** contient l'historique mensuel de tous les prêts chez Credit Bureau (CrB). Chaque ligne correspond à l'état mensuel d'un prêt au sein de CrB caractérisée par:
    - SK_ID_BUREAU
    - MONTHS_BALANCE: $\leq 0$ avec 0, le mois courant
    - STATUS: Etat du remboursement à ce jour [0-5,C,X], 0 a 5 pour un retard sur le remboursemnt (plus la valeur est elevée plus il y a du retard avec 0 pas de retard et 5 pour prêt revendu), C pour un prêt cloturé et X pour dire statut inconnu. 

